In [21]:
# # Install Requirements
import sys
!{sys.executable} -m pip install scikit-learn pandas ray


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: /Users/shreycshah/Desktop/Coursework/Spring26/IE7374/Labs/mlops-modeling-ray-lab/.venv/bin/python -m pip install --upgrade pip


In [22]:
# # Distributed Model Training with Ray
#
# In this lab, you will explore how to speed up machine learning model training
# by distributing work across multiple cores using **Ray**. You will:
#
# 1. Train multiple models **sequentially** and measure wall-clock time.
# 2. Train the same models **in parallel** using Ray remote tasks.
# 3. Compare the performance gains from parallelization.
#
# **Dataset:** Ames Housing (via OpenML) — predicting house sale prices.
# **Model:** RandomForestRegressor with varying `n_estimators`.

# Lab Inspiration: https://github.com/raminmohammadi/MLOps/blob/main/Labs/Model_Development/Ray/Ray.ipynb

In [23]:
import time
from operator import itemgetter

import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [24]:
 ## Prepare Dataset
#
# The Ames Housing dataset contains 80 features describing residential homes in
# Ames, Iowa. We will predict `SalePrice`. Since GradientBoosting in sklearn
# does not natively handle categorical features, we label-encode them first.

# %%
# Fetch Ames Housing dataset from OpenML
ames = fetch_openml(name="house_prices", version=1, as_frame=True, parser="auto")
X = ames.data
y = ames.target

In [25]:
# Label-encode categorical columns
label_encoders = {}
for col in X.select_dtypes(include=["object", "category"]).columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

/var/folders/xv/cg9l3dzd2jl77sg57m_lqh800000gn/T/ipykernel_12215/3372788928.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in X.select_dtypes(include=["object", "category"]).columns:


In [27]:
# Drop columns with any remaining NaN for simplicity
X = X.dropna(axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Test set:     {X_test.shape[0]} samples")
X_train.head(n=5)

Training set: 1168 samples, 77 features
Test set:     292 samples


,Id,MSSubClass,MSZoning,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
254,255,20,3,8400,1,2,3,3,0,4,...,0,0,3,4,4,0,6,2010,8,4
1066,1067,60,3,7837,1,2,0,3,0,4,...,0,0,3,4,4,0,5,2009,8,4
638,639,30,3,8777,1,2,3,3,0,4,...,0,0,3,2,4,0,5,2008,8,4
799,800,50,3,7200,1,2,3,3,0,0,...,0,0,3,2,4,0,6,2007,8,4
380,381,50,3,5000,1,1,3,3,0,4,...,0,0,3,4,4,0,5,2010,8,4


In [28]:
 ## Sequential Implementation
#
# We define `NUM_MODELS` configurations to train. Each model uses a different
# value of `n_estimators` (increasing by 20 each time: 50, 70, 90, ...).
# This simulates a hyperparameter sweep over model complexity.
#
# **Important:** We set `n_jobs=1` in each RandomForest so that sklearn does
# NOT internally parallelize tree building. This ensures that the sequential
# run is truly single-threaded, making the Ray speedup clearly visible.

# %%
# Number of model configurations to train
NUM_MODELS = 20

In [29]:
def train_and_score_model(
    train_set: pd.DataFrame,
    test_set: pd.DataFrame,
    train_labels: pd.Series,
    test_labels: pd.Series,
    n_estimators: int,
) -> tuple[int, float]:
    """Train a RandomForestRegressor and return (n_estimators, RMSE)."""
    start_time = time.time()

    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=12,
        n_jobs=1,  # single-threaded to show Ray's benefit
        random_state=42,
    )
    model.fit(train_set, train_labels)
    y_pred = model.predict(test_set)
    rmse = root_mean_squared_error(test_labels, y_pred)

    time_delta = time.time() - start_time
    print(
        f"n_estimators={n_estimators}, rmse={rmse:.2f}, took: {time_delta:.2f} seconds"
    )
    return n_estimators, rmse

In [30]:
# This function trains `n_models` sequentially for an increasing number of
# `n_estimators` (increasing by 25 for each model, e.g. 50, 75, 100, 125, ...).
#
# `run_sequential` returns a list of tuples:
# ```
# [(n_estimators, rmse_score), (n_estimators, rmse_score), ...]
# ```
# For example:
# ```
# [(50, 29834.12), (75, 27102.45), (100, 26011.33), ...]

def run_sequential(n_models: int) -> list[tuple[int, float]]:
    """Train n_models sequentially with increasing n_estimators."""
    return [
        train_and_score_model(
            train_set=X_train,
            test_set=X_test,
            train_labels=y_train,
            test_labels=y_test,
            n_estimators=50 + 25 * j,
        )
        for j in range(n_models)
    ]

In [31]:
%%time
mse_scores = run_sequential(n_models=NUM_MODELS)

n_estimators=50, rmse=28944.34, took: 0.48 seconds
n_estimators=75, rmse=28430.52, took: 0.61 seconds
n_estimators=100, rmse=28444.58, took: 0.81 seconds
n_estimators=125, rmse=28374.46, took: 1.00 seconds
n_estimators=150, rmse=28177.50, took: 1.22 seconds
n_estimators=175, rmse=28344.33, took: 1.42 seconds
n_estimators=200, rmse=28414.26, took: 1.58 seconds
n_estimators=225, rmse=28291.48, took: 1.78 seconds
n_estimators=250, rmse=28308.09, took: 1.98 seconds
n_estimators=275, rmse=28290.43, took: 2.30 seconds
n_estimators=300, rmse=28252.38, took: 2.42 seconds
n_estimators=325, rmse=28288.56, took: 2.62 seconds
n_estimators=350, rmse=28379.97, took: 2.81 seconds
n_estimators=375, rmse=28395.34, took: 3.22 seconds
n_estimators=400, rmse=28299.95, took: 3.48 seconds
n_estimators=425, rmse=28398.35, took: 3.54 seconds
n_estimators=450, rmse=28401.04, took: 3.75 seconds
n_estimators=475, rmse=28335.84, took: 3.84 seconds
n_estimators=500, rmse=28389.78, took: 3.99 seconds
n_estimators=5

In [32]:
# ### Analyze Sequential Results
best = min(mse_scores, key=itemgetter(1))
print(f"Best model: rmse={best[1]:.2f}, n_estimators={best[0]}")

Best model: rmse=28177.50, n_estimators=150


In [35]:
# Looking at the results of training, make a note of how long training
# `NUM_MODELS` sequentially took. Continue to the next section to learn how
# to improve runtime by distributing this task with Ray.

In [36]:
# ## Parallel Implementation
#
# In contrast to the previous approach, you will now utilize all available
# resources to train these models **in parallel**. Ray will automatically detect
# the number of cores on your computer or the amount of resources in a cluster
# to distribute each defined task.
#
# The key idea: a **scheduler** manages incoming requests, assigns workers
# (one per CPU core), and detects available resources. Instead of training
# models one after another, Ray dispatches all 20 training jobs at once and
# workers pick them up as they become free.

In [37]:
import ray

if ray.is_initialized():
    ray.shutdown()

2026-04-07 09:10:55,046	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [38]:
# After running `ray.init()`, note the reported:
# - Python version
# - Ray version
# - Link to the **Ray Dashboard** — an observability tool that provides insight
#   into what Ray is doing via helpful metrics and charts.
#
# ### Ray Object Store
#
# Workers use `ray.put()` to place objects and `ray.get()` to retrieve them
# from each node's object store. These object stores form the **shared
# distributed memory** that makes objects available across a Ray cluster.
#
# In a distributed system, **object references** are pointers to objects in
# memory. They can be used to access objects stored on different machines,
# allowing workers to communicate and share data.

In [39]:
X_train_ref = ray.put(X_train)
X_test_ref = ray.put(X_test)
y_train_ref = ray.put(y_train)
y_test_ref = ray.put(y_test)

2026-04-07 09:11:00,261	INFO worker.py:2013 -- Started a local Ray instance.
/Users/shreycshah/Desktop/Coursework/Spring26/IE7374/Labs/mlops-modeling-ray-lab/.venv/lib/python3.11/site-packages/ray/_private/worker.py:2052: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


In [40]:
# By placing the training and testing data into Ray's object store, these
# objects are now available to all remote tasks and actors in the cluster.

# print the object reference
print(X_train_ref)

# inspect the in-memory object
ray.get(X_train_ref)

ObjectRef(00ffffffffffffffffffffffffffffffffffffff0100000001e1f505)


,Id,MSSubClass,MSZoning,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
254,255,20,3,8400,1,2,3,3,0,4,...,0,0,3,4,4,0,6,2010,8,4
1066,1067,60,3,7837,1,2,0,3,0,4,...,0,0,3,4,4,0,5,2009,8,4
638,639,30,3,8777,1,2,3,3,0,4,...,0,0,3,2,4,0,5,2008,8,4
799,800,50,3,7200,1,2,3,3,0,0,...,0,0,3,2,4,0,6,2007,8,4
380,381,50,3,5000,1,1,3,3,0,4,...,0,0,3,4,4,0,5,2010,8,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1095,1096,20,3,9317,1,2,0,3,0,4,...,0,0,3,4,4,0,3,2007,8,4
1130,1131,50,3,7804,1,2,3,3,0,4,...,0,0,3,2,4,0,12,2009,8,4
1294,1295,20,3,8172,1,2,3,3,0,4,...,0,0,3,4,4,0,4,2006,8,4
860,861,50,3,7642,1,2,3,3,0,0,...,0,0,3,0,4,0,6,2007,8,4


In [41]:
# Implementation of Remote Function to Train and Score Model
#
# Notice that `train_and_score_model` below is the **same function** as in the
# sequential example, except here you add the `@ray.remote` decorator to
# specify that this function will be executed in a distributed manner.

# %%
@ray.remote
def train_and_score_model(
    train_set_ref: pd.DataFrame,
    test_set_ref: pd.DataFrame,
    train_labels_ref: pd.Series,
    test_labels_ref: pd.Series,
    n_estimators: int,
) -> tuple[int, float]:
    """Ray remote task: Train a GradientBoostingRegressor and return (n_estimators, RMSE)."""
    start_time = time.time()

    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=12,
        n_jobs=1,  # single-threaded per worker
        random_state=42,
    )
    model.fit(train_set_ref, train_labels_ref)
    y_pred = model.predict(test_set_ref)
    rmse = root_mean_squared_error(test_labels_ref, y_pred)

    time_delta = time.time() - start_time
    print(
        f"n_estimators={n_estimators}, rmse={rmse:.2f}, took: {time_delta:.2f} seconds"
    )
    return n_estimators, rmse

In [42]:
### Implement Function That Runs Parallel Model Training
#
# Working from the inside-out, modifying `run_sequential()` into
# `run_parallel()` involves three steps:
#
# 1. **Append `.remote`** to `train_and_score_model`.
#    Since you specified this function as a remote task with `@ray.remote`,
#    you must append `.remote()` to every call.
#
# 2. **Capture the resulting list of object references** in `results_ref`.
#    Rather than waiting for results, you immediately receive references to
#    results expected to be available in the future. This **asynchronous
#    (non-blocking)** call lets the program continue while time-consuming
#    operations compute in the background.
#
# 3. **Access results with `ray.get()`**.
#    Once all models have been assigned to workers, call `ray.get()` on the
#    list of object references to retrieve completed results. This is a
#    **synchronous (blocking)** operation — it waits until all computation
#    finishes.
#
# For example:
# ```python
# ray.get([ObjectRef, ObjectRef, ObjectRef, ...])
# # returns list of (n_estimators, rmse) tuples.
# ```

def run_parallel(n_models: int) -> list[tuple[int, float]]:
    """Train n_models in parallel using Ray remote tasks."""
    results_ref = [
        train_and_score_model.remote(
            train_set_ref=X_train_ref,
            test_set_ref=X_test_ref,
            train_labels_ref=y_train_ref,
            test_labels_ref=y_test_ref,
            n_estimators=50 + 25 * j,
        )
        for j in range(n_models)
    ]
    return ray.get(results_ref)

In [43]:
%%time
mse_scores = run_parallel(n_models=NUM_MODELS)

(train_and_score_model pid=13527) n_estimators=50, rmse=28944.34, took: 0.53 seconds
(train_and_score_model pid=13529) n_estimators=350, rmse=28379.97, took: 3.73 seconds [repeated 12x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)
CPU times: user 33.2 ms, sys: 47.5 ms, total: 80.7 ms
Wall time: 14 s


In [44]:
### Analyze Parallel Results

best = min(mse_scores, key=itemgetter(1))
print(f"Best model: rmse={best[1]:.2f}, n_estimators={best[0]}")

Best model: rmse=28177.50, n_estimators=150
(train_and_score_model pid=13531) n_estimators=525, rmse=28343.54, took: 4.69 seconds


In [45]:
### Performance Comparison
#
# Compare the wall time from the sequential run vs. the parallel run.
# You should observe a significant speedup — the exact factor depends on the
# number of CPU cores available on your machine.
#
# On a machine with N cores, you can expect roughly an **N× speedup** for
# embarrassingly parallel workloads like independent model training.

# %%
# Shutdown Ray runtime
ray.shutdown()